# Step 2 - Preparing the DocLayNet v1.1 pretraining corpus

This notebook investigates and prepares a small, reproducible document-image corpus for the unified text-image masking project. It uses `docling-project/DocLayNet-v1.1` with Hugging Face streaming and does not download the complete dataset.

The standard record contains page images, layout-region annotations, and digital PDF text cells. PDF cells are not assumed to be OCR words. The notebook preserves that distinction when writing metadata for later MLM, MIM, and word-patch alignment work.

**Safety gate:** the smoke test is enabled by default; 200-page extraction is disabled until `RUN_EXTRACTION` is explicitly changed to `True`. If remote streaming fails, the notebook reports the error and does not fall back to a non-streaming/full-download load.

## 1. Objectives

- Verify the DocLayNet v1.1 streaming schema and representative records.
- Inspect page images, layout boxes, and nested `pdf_cells`.
- Select 200 train pages with a fixed seed for pretraining.
- Save PNG images and a metadata CSV under `data/pretraining_corpus/`.
- Validate paths, image dimensions, annotation counts, and provenance.

The corpus is intended to provide images and layout/text context for later LayoutLMv3-style MLM, MIM, and WPA experiments. This notebook does not implement or train those objectives.

## 2. Required libraries

Run this cell only in the notebook environment. The project `requirements.txt` is intentionally not changed by this notebook.

In [ ]:
!pip install -q datasets pillow pandas matplotlib

In [ ]:
from pathlib import Path
import json
import random

import matplotlib.pyplot as plt
import pandas as pd
from datasets import get_dataset_split_names, load_dataset
from PIL import Image, ImageDraw

DATASET_NAME = "docling-project/DocLayNet-v1.1"
DATASET_REVISION = "5e89392376049f4d589ea339ed64468310ed5c3f"
SEED = 20241106
RUN_SMOKE_TEST = True
RUN_EXTRACTION = False
SPLIT_TARGETS = {"train": 200}

CORPUS_ROOT = Path("data/pretraining_corpus")
IMAGE_ROOT = CORPUS_ROOT / "images"
CSV_PATH = CORPUS_ROOT / "page_ids.csv"
BBOX_COORDINATE_SYSTEM = "1025x1025 COCO image pixels; [x, y, width, height]"

random.seed(SEED)
print(f"Dataset: {DATASET_NAME}")
print(f"Seed: {SEED}")
print(f"Extraction enabled: {RUN_EXTRACTION}")

## 3. Dataset information and sources

DocLayNet v1.1 contains 80,863 pages from six document categories. Its official splits are 69,375 train pages, 6,489 validation pages, and 4,999 test pages. The Hugging Face v1.1 record uses `val` for the validation split. This pretraining corpus uses train pages only.

Sources:

- [Hugging Face dataset card](https://huggingface.co/datasets/docling-project/DocLayNet-v1.1)
- [Official DocLayNet repository](https://github.com/DS4SD/DocLayNet)
- [DocLayNet paper](https://doi.org/10.1145/3534678.3539043)
- License: [CDLA-Permissive-1.0](https://cdla.io/permissive-1-0/)

The Parquet files are remote shards. Streaming can avoid materializing the complete dataset, but it may still transfer substantial row groups or shard data. The extraction code therefore never uses a non-streaming fallback.

This notebook pins the verified v1.1 revision `5e89392376049f4d589ea339ed64468310ed5c3f`.

## 4. Loading DocLayNet v1.1 with streaming

If this cell fails, check network access, the dataset name, and the installed `datasets` version. Do not remove `streaming=True` to work around the failure: a non-streaming load can download the complete dataset.

In [ ]:
def streaming_split(split):
    kwargs = {
        "path": DATASET_NAME,
        "split": split,
        "streaming": True,
    }
    if DATASET_REVISION:
        kwargs["revision"] = DATASET_REVISION
    try:
        return load_dataset(**kwargs)
    except Exception as exc:
        raise RuntimeError(
            "DocLayNet streaming could not be opened. No full-download fallback "
            "was attempted. Try the Colab smoke-test cell in the project notes, "
            "or provide a pinned Parquet mirror before extraction."
        ) from exc

split_kwargs = {"path": DATASET_NAME}
if DATASET_REVISION:
    split_kwargs["revision"] = DATASET_REVISION
try:
    available_splits = get_dataset_split_names(**split_kwargs)
except TypeError:
    available_splits = get_dataset_split_names(DATASET_NAME)
print("Available splits:", available_splits)
for split in SPLIT_TARGETS:
    if split not in available_splits:
        raise ValueError(f"Expected split {split!r} is unavailable")
print("Streaming split configuration is available for train, val, and test.")

## 5. Inspecting train, val, and test splits

The expected v1.1 counts are train 69,375, val 6,489, and test 4,999. A streaming dataset exposes metadata and records incrementally; inspecting split names does not download all pages.

In [ ]:
for split in SPLIT_TARGETS:
    stream = streaming_split(split)
    print(split, "features:", stream.features)

## 6. Smoke test: inspect three actual records

This is the only record-loading section enabled by default. It stops after three records and prints bounded examples from the nested annotations.

In [ ]:
def print_record_summary(record, split, index):
    image = record["image"]
    metadata = record["metadata"]
    bboxes = record["bboxes"]
    pdf_cells = record["pdf_cells"]

    print(f"\n{split} record {index}")
    print("keys:", list(record.keys()))
    print("image:", image.mode, image.size)
    print("metadata:", metadata)
    print("layout boxes:", len(bboxes))
    print("pdf cell groups:", len(pdf_cells))

    shown = 0
    for group_index, group in enumerate(pdf_cells):
        for cell in group:
            print("cell", {"group": group_index, "bbox": cell["bbox"], "text": cell["text"][:160]})
            shown += 1
            if shown == 3:
                return

if RUN_SMOKE_TEST:
    smoke_stream = streaming_split("train")
    for index, record in enumerate(smoke_stream.take(3)):
        print_record_summary(record, "train", index)

## 7. Image dimensions and formats

DocLayNet v1.1 provides page images at 1025 x 1025 pixels. The extraction converts each image to RGB before saving PNG, while retaining the source dimensions in the CSV.

In [ ]:
if RUN_SMOKE_TEST:
    smoke_stream = streaming_split("train")
    for index, record in enumerate(smoke_stream.take(3)):
        image = record["image"]
        print(index, image.format, image.mode, image.size)

## 8. Layout bounding boxes

`bboxes` are layout-region boxes, not OCR word boxes. They use `[x, y, width, height]` in the 1025 x 1025 COCO image coordinate system. LayoutLMv3 normalization to 0-1000 is a future preprocessing step. `category_id` uses the 11 DocLayNet classes: Caption, Footnote, Formula, List-item, Page-footer, Page-header, Picture, Section-header, Table, Text, and Title.

In [ ]:
CATEGORY_NAMES = {
    1: "Caption", 2: "Footnote", 3: "Formula", 4: "List-item",
    5: "Page-footer", 6: "Page-header", 7: "Picture",
    8: "Section-header", 9: "Table", 10: "Text", 11: "Title",
}

if RUN_SMOKE_TEST:
    record = next(iter(streaming_split("train").take(1)))
    print("First five layout boxes:")
    for box, category_id in list(zip(record["bboxes"], record["category_id"]))[:5]:
        print({"bbox": box, "category_id": category_id, "category": CATEGORY_NAMES.get(category_id)})

## 9. `pdf_cells` and nested structure

`pdf_cells` is a list of lists. The outer list corresponds to layout regions, and each inner list contains digital PDF text cells associated with that region. A cell contains `bbox`, `font`, and `text`. The text can be a heading, a complete line, a paragraph fragment, or a table value.

In [ ]:
if RUN_SMOKE_TEST:
    record = next(iter(streaming_split("train").take(1)))
    for group_index, group in enumerate(record["pdf_cells"][:3]):
        print("group", group_index, "cell_count", len(group))
        for cell in group[:3]:
            print({"bbox": cell["bbox"], "text": cell["text"], "font": cell["font"]})

## 10. Why `pdf_cells` are not necessarily OCR words

These annotations come from digital PDF text extraction, not an OCR word detector. A cell may contain an entire sentence or line, and scanned or image-only content may have no digital text cell. Consequently, this notebook sets `has_word_level_ocr` to `False`. Later preprocessing must tokenize cell text explicitly and must not claim that region or cell boxes are original word-level boxes.

## 11. Visualizing layout boxes

The following visualization uses one smoke-test image and draws the first layout boxes. It does not write an image to the corpus.

In [ ]:
if RUN_SMOKE_TEST:
    record = next(iter(streaming_split("train").take(1)))
    image = record["image"].convert("RGB")
    preview = image.copy()
    draw = ImageDraw.Draw(preview)
    for box, category_id in zip(record["bboxes"][:30], record["category_id"][:30]):
        x, y, width, height = box
        draw.rectangle((x, y, x + width, y + height), outline="red", width=2)
        draw.text((x, y), str(category_id), fill="blue")
    plt.figure(figsize=(8, 8))
    plt.imshow(preview)
    plt.axis("off")
    plt.show()

## 12. Reproducible selection of approximately 200 pages

The extraction uses a fixed seed and a bounded streaming shuffle to select 200 train pages. Validation and test pages are intentionally excluded from this pretraining corpus. The shuffle buffer is finite, so this is reproducible for the pinned dataset revision and library behavior, but it is not equivalent to a full in-memory random shuffle. The code stops after the requested records and does not load the complete dataset.

In [ ]:
def select_records():
    selected = []
    for split, target in SPLIT_TARGETS.items():
        stream = streaming_split(split).shuffle(seed=SEED, buffer_size=10_000)
        for record in stream.take(target):
            selected.append((split, record))
    return selected

if RUN_EXTRACTION:
    print("Streaming selection begins. Remote Parquet may transfer more than 200 records.")
    selected_records = select_records()
    print("Selected records:", len(selected_records))
else:
    selected_records = []
    print("Extraction is disabled. Set RUN_EXTRACTION = True only after review.")

## 13. Saving PNG images

Images are named from the original `image_id`. The source image is converted to RGB and saved as PNG. Existing matching files are reused without overwriting; mismatched existing files raise an error. Each image is first written to a temporary path and then atomically renamed, so a failed write does not leave a partial PNG.

In [ ]:
def serialize_cells(pdf_cells):
    return json.dumps(pdf_cells, ensure_ascii=True, separators=(",", ":"))

def save_image_safely(image, image_path):
    if image_path.exists():
        with Image.open(image_path) as existing:
            if existing.format != "PNG" or existing.mode != "RGB" or existing.size != image.size:
                raise ValueError(f"Existing image does not match expected PNG: {image_path}")
        return
    temporary_path = image_path.with_name(image_path.name + ".tmp")
    try:
        image.save(temporary_path, format="PNG")
        temporary_path.replace(image_path)
    finally:
        if temporary_path.exists():
            temporary_path.unlink()

def save_selected_records(records):
    IMAGE_ROOT.mkdir(parents=True, exist_ok=True)
    rows = []
    for split, record in records:
        metadata = record["metadata"]
        image_id = metadata["image_id"]
        filename = f"doclaynet_v1_1_{split}_{image_id}.png"
        image_path = IMAGE_ROOT / filename
        image = record["image"].convert("RGB")
        save_image_safely(image, image_path)
        pdf_cells = record["pdf_cells"]
        rows.append({
            "page_id": f"doclaynet-v1.1-{split}-{image_id}",
            "dataset": DATASET_NAME,
            "split": split,
            "image_id": image_id,
            "doc_name": metadata["original_filename"],
            "page_no": metadata["page_no"],
            "doc_category": metadata["doc_category"],
            "image_path": str(image_path).replace("\\", "/"),
            "width": image.width,
            "height": image.height,
            "num_layout_boxes": len(record["bboxes"]),
            "num_pdf_cells": sum(len(group) for group in pdf_cells),
            "has_digital_text": bool(pdf_cells and any(pdf_cells)),
            "has_word_level_ocr": False,
            "bbox_coordinate_system": BBOX_COORDINATE_SYSTEM,
            "sampling_seed": SEED,
            "layout_boxes_json": json.dumps(record["bboxes"], separators=(",", ":")),
            "layout_categories_json": json.dumps(record["category_id"], separators=(",", ":")),
            "pdf_cells_json": serialize_cells(pdf_cells),
            "segmentation_json": json.dumps(record.get("segmentation", []), separators=(",", ":")),
            "area_json": json.dumps(record.get("area", []), separators=(",", ":")),
            "page_hash": metadata["page_hash"],
            "collection": metadata["collection"],
            "num_pages": metadata["num_pages"],
            "source_revision": DATASET_REVISION or "un-pinned-main-at-run-time",
        })
    frame = pd.DataFrame(rows)
    CORPUS_ROOT.mkdir(parents=True, exist_ok=True)
    frame.to_csv(CSV_PATH, index=False)
    return frame

if RUN_EXTRACTION:
    corpus_df = save_selected_records(selected_records)
    print(f"Saved {len(corpus_df)} rows to {CSV_PATH}")

## 14. Creating and inspecting `page_ids.csv`

The required columns are included below. The additional JSON columns preserve the layout boxes and digital PDF cells without relabeling them as OCR words.

In [ ]:
REQUIRED_COLUMNS = [
    "page_id", "dataset", "split", "image_id", "doc_name", "page_no",
    "doc_category", "image_path", "width", "height", "num_layout_boxes",
    "num_pdf_cells", "has_digital_text", "has_word_level_ocr",
    "bbox_coordinate_system", "sampling_seed",
]

if RUN_EXTRACTION:
    missing = [column for column in REQUIRED_COLUMNS if column not in corpus_df.columns]
    if missing:
        raise AssertionError(f"Missing CSV columns: {missing}")
    display(corpus_df[REQUIRED_COLUMNS].head())

## 15. Validating saved images and CSV

Validation checks that every CSV image exists, opens as RGB PNG, has the recorded dimensions, and has consistent annotation flags. It does not run unless extraction has been explicitly enabled.

In [ ]:
def validate_corpus(frame):
    assert len(frame) == sum(SPLIT_TARGETS.values())
    assert frame["page_id"].is_unique
    assert set(frame["dataset"]) == {DATASET_NAME}
    assert set(frame["has_word_level_ocr"]) == {False}
    for row in frame.to_dict("records"):
        path = Path(row["image_path"])
        assert path.exists(), path
        with Image.open(path) as image:
            assert image.format == "PNG"
            assert image.mode == "RGB"
            assert image.size == (row["width"], row["height"])
    print("Validation passed for", len(frame), "pages")

if RUN_EXTRACTION:
    validate_corpus(corpus_df)


## 16. Final corpus summary

The summary reports the selected counts, image dimensions, document categories, and digital-text availability. A successful summary means the extraction and validation cells were explicitly enabled and completed; it is not produced by the default smoke-test run.

In [ ]:
if RUN_EXTRACTION:
    print("Dataset:", DATASET_NAME)
    print("Pages:", len(corpus_df))
    print("By split:")
    print(corpus_df["split"].value_counts().sort_index())
    print("By category:")
    print(corpus_df["doc_category"].value_counts().sort_index())
    print("Images with digital PDF text:", int(corpus_df["has_digital_text"].sum()))
    print("Word-level OCR records:", int(corpus_df["has_word_level_ocr"].sum()))
    print("CSV:", CSV_PATH)
    print("Images:", IMAGE_ROOT)
else:
    print("No corpus images were downloaded. Smoke test only.")